In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS demo;

In [0]:
SHOW DATABASES

databaseName
default
demo
information_schema


In [0]:
DESCRIBE DATABASE EXTENDED demo

database_description_item,database_description_value
Catalog Name,databricks_course_centralus_ws
Namespace Name,demo
Comment,
Location,
Owner,gonzalosanchezyallico@hotmail.com
Properties,"((unity.catalog.managed.delta.defaults.defaultClusterByAuto,true), (unity.catalog.managed.delta.defaults.delta.checkpointPolicy,v2), (unity.catalog.managed.delta.defaults.delta.enableRowTracking,true), (unity.catalog.managed.delta.defaults.delta.parquet.format.version,2.12.0), (unity.catalog.managed.iceberg.defaults.delta.feature.catalogManaged,supported))"
Predictive Optimization,ENABLE (inherited from METASTORE metastore_azure_centralus)


In [0]:
SELECT current_database()

current_schema()
default


### Tablas administradas (managed tables)

In [0]:
%run '../includes/configuration'

In [0]:
%python
results_movie_genre_language = spark.read.parquet(f"{gold_folder_path}/results_movie_genre_language")

In [0]:
USE demo

In [0]:
%python
results_movie_genre_language.write.format("delta").saveAsTable("results_movie_genre_language_python")

In [0]:
SHOW TABLES

database,tableName,isTemporary
demo,results_movie_genre_language_python,false


### Trabajando con tablas externas

In [0]:
%python
results_movie_genre_language.write.format("delta").mode("overwrite").option("path", f"{gold_folder_path}/results_movie_genre_language_py").saveAsTable("demo.results_movie_genre_language_py")

In [0]:
DESC EXTENDED demo.results_movie_genre_language_python_py

col_name,data_type,comment
title,string,null
duration_time,int,null
release_date,date,null
vote_average,double,null
language_name,string,null
genre_name,string,null
created_date,timestamp,null
,,
# Delta Statistics Columns,,
Column Names,"duration_time, created_date, vote_average, release_date, genre_name, title, language_name",


In [0]:
DROP TABLE results_movie_genre_language_python_py

In [0]:
SHOW TABLES IN demo

database,tableName,isTemporary
demo,results_movie_genre_language_py,false
demo,results_movie_genre_language_python,false


In [0]:
CREATE TABLE demo.results_movie_genre_language_sql(
    title STRING,
    duration_time INT,
    release_date DATE,
    vote_average FLOAT,
    language_name STRING,
    genre_name STRING,
    created_date TIMESTAMP
)
USING DELTA
LOCATION "abfss://gold@moviehistoryg.dfs.core.windows.net/results_movie_genre_language_ext_sql"

In [0]:
SHOW TABLES IN demo

database,tableName,isTemporary
demo,results_movie_genre_language_py,false
demo,results_movie_genre_language_python,false
demo,results_movie_genre_language_sql,false


In [0]:
select* from demo.results_movie_genre_language_sql

title,duration_time,release_date,vote_average,language_name,genre_name,created_date


### Vistas(Views)

In [0]:
SELECT current_database()

current_schema()
demo


In [0]:
CREATE OR REPLACE TEMP VIEW v_results_movies_genres_language
AS
SELECT * FROM
demo.results_movie_genre_language_py
WHERE genre_name = 'Adventure'

In [0]:
SELECT * FROM v_results_movies_genres_language

title,duration_time,release_date,vote_average,language_name,genre_name,created_date
Kicks,80,2016-09-09,7.5,English,Adventure,2026-10-29T04:20:24.911404Z
Kicks,80,2016-09-09,7.5,English,Adventure,2026-10-29T04:20:24.911404Z
Pete's Dragon,102,2016-08-10,6.2,English,Adventure,2026-10-29T04:20:24.911404Z
Pete's Dragon,102,2016-08-10,6.2,English,Adventure,2026-10-29T04:20:24.911404Z
Suicide Squad,123,2016-08-02,5.9,English,Adventure,2026-10-29T04:20:24.911404Z
Suicide Squad,123,2016-08-02,5.9,English,Adventure,2026-10-29T04:20:24.911404Z
Nerve,96,2016-07-27,7.1,English,Adventure,2026-10-29T04:20:24.911404Z
Nerve,96,2016-07-27,7.1,English,Adventure,2026-10-29T04:20:24.911404Z
Sausage Party,83,2016-07-11,5.6,English,Adventure,2026-10-29T04:20:24.911404Z
Sausage Party,83,2016-07-11,5.6,English,Adventure,2026-10-29T04:20:24.911404Z


In [0]:
CREATE OR REPLACE GLOBAL TEMP VIEW gv_results_movies_genres_language
AS
SELECT * FROM
demo.results_movie_genre_language_py
WHERE genre_name = 'Drama';

In [0]:
SHOW TABLES ;

database,tableName,isTemporary
demo,results_movie_genre_language_py,false
demo,results_movie_genre_language_python,false
demo,results_movie_genre_language_sql,false
,v_results_movies_genres_language,true


In [0]:
SELECT * FROM global_temp.gv_results_movies_genres_language 

title,duration_time,release_date,vote_average,language_name,genre_name,created_date
Growing Up Smith,102,2017-02-03,7.4,English,Drama,2026-10-29T04:20:24.911404Z
Growing Up Smith,102,2017-02-03,7.4,English,Drama,2026-10-29T04:20:24.911404Z
Two Lovers and a Bear,96,2016-10-02,6.8,English,Drama,2026-10-29T04:20:24.911404Z
Two Lovers and a Bear,96,2016-10-02,6.8,English,Drama,2026-10-29T04:20:24.911404Z
Mr. Church,104,2016-09-16,7.0,English,Drama,2026-10-29T04:20:24.911404Z
Mr. Church,104,2016-09-16,7.0,English,Drama,2026-10-29T04:20:24.911404Z
The Birth of a Nation,120,2016-09-09,6.5,English,Drama,2026-10-29T04:20:24.911404Z
The Birth of a Nation,120,2016-09-09,6.5,English,Drama,2026-10-29T04:20:24.911404Z
Hands of Stone,105,2016-08-26,6.1,English,Drama,2026-10-29T04:20:24.911404Z
Hands of Stone,105,2016-08-26,6.1,Español,Drama,2026-10-29T04:20:24.911404Z


In [0]:
CREATE OR REPLACE VIEW pv_results_movies_genres_language
AS
SELECT * FROM
demo.results_movie_genre_language_py
WHERE genre_name = 'Comedy';

In [0]:
SHOW TABLES 

database,tableName,isTemporary
demo,pv_results_movies_genres_language,false
demo,results_movie_genre_language_py,false
demo,results_movie_genre_language_python,false
demo,results_movie_genre_language_sql,false
,v_results_movies_genres_language,true
